# 实验六 · 矩阵乘法 MatMul —— 从矢量单元到 Cube 单元

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐⭐ 进阶　|　**预计时长**：50–60 分钟

矩阵乘法是深度学习中占比最大的一类计算，昇腾为它准备了专门的硬件：**Cube 单元**。这也使它成为本章第一个用哪个计算单元比怎么写这个算子更要紧的实验。

本实验分两步走：

1. **先在矢量单元上把矩阵乘写出来**，并把它优化到这条路径的上限。矩阵乘的每个输入元素可以被反复使用，因此优化的方向与前五个实验不同——不是减少必须搬运的数据，而是提高已搬入数据的复用次数。
2. **再用 CANN 主推的 `Matmul` 高阶 API 把同一个矩阵乘交给 Cube 单元**，对照两者的差距。

这样安排的用意是：**先测出矢量单元的性能上限，再考察 Cube 单元解决的是其中哪一项限制。** 直接给出「矩阵乘应当使用 Cube」这一结论，学生只能记住结论本身；先测出矢量单元的上限，这一结论才有可以依据的事实。

> **实验说明**
> 1. 前半程的三个版本共用**同一个核函数模板**，v1 与 v2 只差一个模板实参，v2 与 v3 只差启动时的核数。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 4. 本实验的规模是 $M = N = K = 1024$，CPU 基准耗时以秒计，请耐心等待。
> 5. 第 12 节的 Cube 版本是一份独立的源文件，由单独一条编译命令生成，需要额外链接 Matmul 高阶 API 的相关库。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说明矩阵乘法与前五个实验的算子在数据复用上的根本差别
- 用秩一更新（外积累加）的形式在矢量单元上实现矩阵乘
- 核算复用系数、片上占用与计算访存比三者的关系，并给出 `TILE_M` 的上界
- 指出矢量单元实现矩阵乘的三项固有限制
- 说出 Cube 单元的片上通路与矢量单元有何不同，以及它逐条消除了上述哪几项限制
- 使用 `Matmul` 高阶 API 在 Cube 单元上实现矩阵乘，说明它的五个调用步骤与各自的作用
- 从 $K$ 与数据分布推导矩阵乘的误差容差，说明它为什么必须随规模缩放
- 说明绝对误差判定与相对误差判定各自的适用条件


## 🗺️ 学习路径

**第一程 · 矢量单元**

1. **准备阶段**：确定算法形式，核算复用、片上占用与计算访存比
2. **v1 · 无复用**：`TILE_M = 1`，$B$ 被完整读 $M$ 遍
3. **v2 · 片上复用**：`TILE_M = 16`，$B$ 的每一行被复用 16 次
4. **v3 · 多核**：沿 $M$ 方向切分到多个矢量核
5. **复用系数扫描**：`TILE_M` 从 1 到 16，看收益在哪一档消失

**第二程 · Cube 单元**

6. **v4 · `Matmul` 高阶 API**：同一个矩阵乘，交给 Cube 单元
7. **分析**：两条路径的差距来自哪里


## 1. 背景与动机：矩阵乘为什么不一样

### 1.1 与前五个实验的根本差别

矩阵乘法定义为

$$ C_{ij} = \sum_{k=0}^{K-1} A_{ik} B_{kj}, \qquad M = N = K = 1024 $$

总运算量是 $2MNK = 2.15$ GFLOP，而三个矩阵合计只有 $3 \times 4$ MiB。**理论上每个字节可以支撑几百次运算**——这是前五个实验的算子做不到的。

官方把单位搬运量对应多少计算这件事称为**计算访存比**，并在 Matmul 的基本块选择中直接用它做判据：

> 基本块的选择原则为计算访存比最大，即在 Cube 计算量最大的情况下，访存的数据量最小。
> ——《Ascend C 算子开发指南》「Matmul 基本块选择」一节

本实验沿用这个概念，取 FLOP/Byte 作单位（在通用文献中它常被称作算术强度）。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| | 实验二至实验五 | 本实验 |
| --- | --- | --- |
| 每个输入元素被用几次 | 1 次 | $A$ 用 $N$ 次，$B$ 用 $M$ 次 |
| 计算访存比 | 由算子固定在 0.5 附近 | **由实现决定，可以从 0.5 调到几百** |
| 优化的方向 | 减少搬运 | 提高复用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">实验二至实验五</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个输入元素被用几次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">A 用 N 次，B 用 M 次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计算访存比</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由算子固定在 0.5 附近</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>由实现决定，可以从 0.5 调到几百</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">优化的方向</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">减少搬运</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">提高复用</td>
</tr>
</tbody>
</table>

但复用不会自动发生。**片上缓冲装不下整个矩阵**，因此必须分块；一旦分块，能复用多少就取决于块开多大。这就是本实验的主线。

### 1.2 秩一更新

在矢量单元上实现矩阵乘，最自然的形式不是逐个算出每一个 $C_{ij}$，而是**秩一更新**：取 $A$ 的第 $k$ 列与 $B$ 的第 $k$ 行做外积，累加到整个 $C$ 上。

$$ C \mathrel{+}= \sum_{k=0}^{K-1} A_{:,k} \otimes B_{k,:} $$

落到代码上就是两层循环：外层遍历 $k$，内层遍历本块的 `TILE_M` 行，每一行做一次向量乘标量再累加。这与第三章 NEON 实验里 AXPY → GEMV → GEMM 的推进路线完全一致，只是把 NEON 的 4 通道换成了长度 $N$ 的 `LocalTensor`。

<img src="./images/06.06_rank1_reuse.png" alt="06.06_rank1_reuse"  width="900px" >

**关键在于 `TILE_M`**：一次把 $C$ 的多少行放在片上一起累加。$B$ 的第 $k$ 行搬进来之后，会被这 `TILE_M` 行同时使用——**`TILE_M` 就是复用系数**。

### 1.3 复用系数、搬运量与计算访存比

设 `TILE_M` = $T$。$C$ 的 $M$ 行被分成 $M/T$ 块，每一块都要把整个 $B$ 读一遍，因此

$$ \text{B 的读取总量} = \frac{M}{T} \cdot K \cdot N \cdot 4 \ \text{字节}, \qquad
   I = \frac{2MNK}{\frac{M}{T} K N \cdot 4} = \frac{T}{2} \ \text{FLOP/Byte} $$

（$A$ 与 $C$ 各只读写一次，合计 8 MiB，相对 $B$ 可以忽略。）

<img src="./images/06.06_traffic_intensity.png" alt="06.06_traffic_intensity"  width="900px" >

**运算量一条指令都没有变，变的只有搬运量。** 提高计算访存比有两条途径：减少必须搬运的数据总量，或提高已搬入数据的复用次数。实验五采用的是前者，本实验采用的是后者。

复用系数 $T$ 能开到多大，第 3 节按片上容量给出上界；开到多大**才有收益**，第 11 节用实测回答。


## 2. 实现要点

### 2.1 三段流水的框架在这里不再适用

实验二至实验五的每个核函数都是「搬入一块 → 算一块 → 搬出一块」。本实验不是：$C$ 的一块要在片上停留整个 $K$ 循环，反复被累加 $K$ 次，最后才搬出去一次。

因此代码的结构变成两层循环：

```text
for 每个 M 块（TILE_M 行）:
    搬入 A 的 TILE_M 行        ← 一次
    把 C 的 TILE_M 行清零      ← 驻留片上
    for k = 0 … K-1:
        搬入 B 的第 k 行        ← 被流水
        for i = 0 … TILE_M-1:
            C[i][:] += A[i][k] · B[k][:]
    搬出 C 的 TILE_M 行        ← 一次
```

只有 $B$ 的搬入是被流水起来的，$A$ 与 $C$ 每块各搬一次。**这个差别直接决定了队列深度的选择**，见 2.3。

### 2.2 标量取值：矢量单元做矩阵乘的固有代价

`A[i][k]` 是一个**标量**，要作为 `Muls` 的乘数使用，必须先用 `GetValue` 从片上缓冲里取出来送进标量寄存器。整个矩阵乘一共要取 $M \times K$ 次，即 100 万次。

每一次 `GetValue` 都在矢量流水与标量流水之间引入一次依赖：标量单元必须等待矢量侧写完数据，取出之后矢量单元才能继续。前五个实验的算子只用到 `Muls`、`Adds` 这类把标量直接写在指令中的形式，不涉及这种同步。

> 这是 Cube 单元存在的理由之一：矩阵乘所需的标量乘向量再累加是一种固定模式，把它做成专用硬件，就不必在通用矢量单元上反复付出这笔同步代价。第 12 节会看到 Cube 的通路上根本没有标量参与。

### 2.3 队列深度不再统一取 2

前五个实验的每个 `TQue` 都用深度 2 做双缓冲。本实验不能这样：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 缓冲 | 内容 | 搬运频率 | 队列深度 | 理由 |
| --- | --- | --- | --- | --- |
| `inQueueA` | `TILE_M` 行 $A$ | 每个 M 块一次 | **1** | 深度 2 会让 $A$ 的占用翻倍，片上放不下 |
| `outQueueC` | `TILE_M` 行 $C$ | 每个 M 块一次 | **1** | 同上 |
| `inQueueB` | $B$ 的一行 | 每个 $k$ 一次 | **2** | 这是唯一被流水的搬运，双缓冲用在这里才有意义 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">缓冲</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">搬运频率</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">队列深度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">理由</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>inQueueA</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TILE_M</code> 行 A</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 M 块一次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">深度 2 会让 A 的占用翻倍，片上放不下</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>outQueueC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TILE_M</code> 行 C</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 M 块一次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同上</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>inQueueB</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">B 的一行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 k 一次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">这是唯一被流水的搬运，双缓冲用在这里才有意义</td>
</tr>
</tbody>
</table>

**双缓冲不是越多越好，它应当用在真正被流水起来的那一段搬运上。**

### 2.4 队列仍然负责同步

`inQueueA` 每个 M 块只搬一次，看上去像是可以用 `TBuf`。**不可以**：`DataCopy` 走 MTE2 流水，`Muls` 走矢量流水，两者之间必须有同步，而 `TBuf` 不提供同步。因此代码里会出现「`EnQue` 之后紧接着 `DeQue`」的写法——它的作用不是排队，而是**取得那一次 MTE2 → 矢量的同步**。

## 3. 片上占用核算与 `TILE_M` 的上界

按 2.3 的深度选择逐项核算（$N = K = 1024$，float32）：

$$ \text{UB} = \underbrace{T \cdot K \cdot 4}_{A} + \underbrace{T \cdot N \cdot 4}_{C}
   + \underbrace{2 \cdot N \cdot 4}_{B} + \underbrace{N \cdot 4}_{\text{tmpBuf}} = 8T + 12 \ \text{KB} $$

<img src="./images/06.06_ub_budget.png" alt="06.06_ub_budget"  width="900px" >

**`TILE_M` 的上界是 22**：$8 \times 22 + 12 = 188$ KB，占 UB 的 98%；再往上就放不下了。

由此得到计算访存比的一个上界：**$I \le T_{\max}/2 = 11$ FLOP/Byte，由片上容量决定。** 要越过这个上界，只能改变数据布局或换用别的计算单元——第 12 节走的正是后一条路。

需要强调的是，这里得到的是**上界**，而不是最优取值。上界由片上容量决定；实际的最佳复用系数则由搬运与计算两条流水的相对速度决定。两者不是同一件事，后者要由第 11 节的扫描测定。

> 与实验二对照：那里增大 `TILE_LENGTH` 带来的是分块数减少，收益很快饱和；这里增大 `TILE_M` 带来的是**搬运总量成比例下降**。同样是增大分块，在两个实验中的含义并不相同。

## 4. 本实验的测量方法

### 4.1 两个口径

`cpu_ms` 与 `kernel_ms` 两个口径与前几个实验一致，计时三纪律同样不再复述。**不设端到端口径**：理由与实验五第 11 节 ② 相同——判断一次卸载是否划算，单位应当是整段计算图，而不是其中的某一个算子。

本实验额外报告 **GFLOPS**：

$$ \text{GFLOPS} = \frac{2MNK}{10^9 \cdot t} $$

从本实验开始，**加速比不再是唯一的评价指标**。加速比只能说明比某个基准快多少，GFLOPS 则可以直接与硬件能力对照，回答离硬件的上限还有多远——第 12 节把矢量单元与 Cube 单元放在一起比较，用的正是这个口径。

### 4.2 两份 CPU 基准

- `RunCpuNaive`：朴素三重循环 $i$-$j$-$k$。内层 $k$ 变化时 $B$ 按列访问，跨步 $N$，局部性最差。
- `RunCpuBlocked`：循环次序换成 $i$-$k$-$j$ 并做分块，与第五章 OpenMP GEMM 的做法一致（此处不开多线程）。

**加速比一律以分块版为基准**，即拿 NPU 与 CPU 上更好的那份实现相比，以免高估收益。朴素版单独报告，它回答的是另一个问题：**同一个算法，仅仅换一下循环次序，在 CPU 上能差多少。**

### 4.3 容差必须随 $K$ 缩放

矩阵乘的每个输出元素是 $K$ 个乘积的累加，误差随累加长度增长，**因此容差不能写成一个与规模无关的常数**。

沿用实验三建立的结论：串行累加链的舍入误差按 $\sqrt{k}$ 增长。第 $j$ 步的部分和量级约为 $\sqrt{j}\,\sigma$，该步引入的舍入误差不超过 $\varepsilon \sqrt{j}\,\sigma$；各步误差符号随机，按平方和累加，得到

$$ \text{误差} \approx \varepsilon \sigma \sqrt{\textstyle\sum_{j=1}^{K} j} \approx \frac{\varepsilon \sigma K}{\sqrt 2}, \qquad \sigma = \mathbb{E}|A_{ik}B_{kj}| $$

由于 $\max|C| \approx \sqrt{K}\,\sigma$，上式可以改写成一个**只用运行期已知量**的形式：

$$ \text{atol} = c \cdot \varepsilon \cdot \sqrt{K} \cdot \max|C| $$

代码取安全系数 $c = 2$。程序会同时打印推导出的 `atol` 与实测的最大绝对误差，二者之比就是余量——容差不是经验常数，它可以推导得出，并且在运行中直接验证。

### 4.4 为什么用绝对误差而不用相对误差

$C_{ij}$ 可以接近 0——那是 $K$ 个正负项相消的结果——但**误差并不随之减小**：它来自那 $K$ 个量级为 $\sigma$ 的乘积项各自的舍入，与相消的程度无关。对一个相消到 $10^{-3}$ 的元素要求相对误差，是没有意义的。

反过来，若误差本身具有相对性质，即输出量级变小时误差按同一比例变小（例如误差由某条近似指令的相对误差引入），那么相对误差才是稳定的判据。

**判定口径应当跟随误差的来源，而不是跟随输出的量级。**

## 5. 环境准备与检查

In [ ]:
!mkdir -p src_matmul

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[
            :1800
        ]
    )

## 6. 版本设计总览

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 计算单元 | 新增的唯一概念 | `TILE_M` | 核数 | 片上占用 | $B$ 的读取总量 | 计算访存比 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| **v1** | 矢量（AIV） | 秩一更新的矢量实现 | **1** | 1 | 20 KB | 4.29 GB | 0.5 |
| **v2** | 矢量（AIV） | 片上复用 | **16** | 1 | 140 KB | 0.27 GB | **8.0** |
| **v3** | 矢量（AIV） | 多核切分 | 16 | **8** | 140 KB | 0.27 GB | 8.0 |
| **v4** | **Cube（AIC）** | **`Matmul` 高阶 API** | 由 Tiling 决定 | 由 Tiling 决定 | 不占 UB | 由 Tiling 决定 | 由 Tiling 决定 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算单元</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">新增的唯一概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">`TILE_M`</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">片上占用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">$B$ 的读取总量</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计算访存比</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**v1**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量（AIV）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">秩一更新的矢量实现</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**1**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">20 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4.29 GB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0.5</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**v2**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量（AIV）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**16**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">140 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0.27 GB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**8.0**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**v3**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量（AIV）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多核切分</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">16</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**8**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">140 KB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0.27 GB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8.0</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**v4**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**Cube（AIC）**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**`Matmul` 高阶 API**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 Tiling 决定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 Tiling 决定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不占 UB</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 Tiling 决定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 Tiling 决定</td>
</tr>
</tbody>
</table>

前三个版本共用同一个核函数模板 `KernelMatmul<TILE_M>`：

- **v1 与 v2 的差别只有模板实参**（1 与 16），源码一行都不必改；
- **v2 与 v3 的差别只有启动时的核数**，连模板实参都相同。

这样安排是为了让两处变化互不干扰：**v1 → v2 单独度量复用的收益，v2 → v3 单独度量并行的收益**。多核切分沿 $M$ 方向进行，每个核负责 $M/\text{blockDim}$ 行，各自读取完整的 $B$——因此 $B$ 的读取总量与 v2 相同，多核不会额外增加搬运。

**v4 换了计算单元**，因此上表最后四列对它不再适用：分块由 `Matmul` 高阶 API 依据 Tiling 决定，数据不经过 UB，开发者也不再直接核算这些量。它单独放在第 12 节，用一份独立的源文件实现。


## 7. Device 侧实现

### 7.1 文件头与参数

与前四个实验一样，全部可调参数用 `#ifndef` 包裹，以便第 11 节与动手练习用 `-D` 在命令行上覆盖。

In [ ]:
%%writefile src_matmul/ascendc_matmul.asc
/**
 * 并行计算 第六章 实验六：矩阵乘法 MatMul（矢量单元实现）
 *
 * 本文件包含一个核函数模板、三个核函数入口、两份 CPU 基准、
 * 数据生成、校验与 main，由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_matmul/ascendc_matmul.asc --npu-arch=dav-2201 -O2 -o src_matmul/ascendc_matmul
 *
 * 本实验的 Host 侧没有用到超越函数，因此不需要 -lm。
 *
 * 用法：
 *   ./ascendc_matmul          三个版本对照，规模取默认值
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>  // 计时：clock_gettime
#include <vector>
#include <algorithm>  // std::fill

#include "acl/acl.h"          // Host 侧
#include "kernel_operator.h"  // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

/* 矩阵规模：C[M, N] = A[M, K] x B[K, N] */
#ifndef LAB_M
#define LAB_M (1024)
#endif
#ifndef LAB_N
#define LAB_N (1024)
#endif
#ifndef LAB_K
#define LAB_K (1024)
#endif
constexpr uint32_t M_SIZE = static_cast<uint32_t>(LAB_M);
constexpr uint32_t N_SIZE = static_cast<uint32_t>(LAB_N);
constexpr uint32_t K_SIZE = static_cast<uint32_t>(LAB_K);

/* 复用系数：一次在片上累加 C 的多少行。上界由 UB 容量决定，见 3 节 */
#ifndef LAB_TILE_M
#define LAB_TILE_M (16)
#endif
constexpr uint32_t TILE_M = static_cast<uint32_t>(LAB_TILE_M);

/* v1 的复用系数固定为 1：不做任何片上复用 */
constexpr uint32_t TILE_M_V1 = 1;

/* v3 使用的核数 */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数。本实验单次核函数是毫秒量级，重复次数取得比前几个实验小 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (10)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 2;

/* 参数扫描时置 1，跳过耗时以秒计的朴素 CPU 基准 */
#ifndef LAB_SKIP_CPU_NAIVE
#define LAB_SKIP_CPU_NAIVE (0)
#endif

/* 只有 B 的搬入是被流水的，因此只有它用双缓冲。理由见 2.3 节 */
constexpr uint32_t QUEUE_DEPTH_B = 2;
constexpr uint32_t QUEUE_DEPTH_AC = 1;

/* float32 的机器精度，用于按 4.3 节的公式推导容差 */
constexpr double F32_EPS = 1.1920928955078125e-7;
constexpr double ATOL_COEF = 2.0; /* 安全系数 */

/* CPU 分块基准的分块参数，与第五章一致的量级 */
constexpr uint32_t CPU_BM = 64;
constexpr uint32_t CPU_BK = 64;
constexpr uint32_t CPU_BN = 256;

### 7.2 核函数模板 `KernelMatmul<TM>`

整个 Device 侧只有这一个类。请对照 2.1 节的伪代码阅读，注意三处与前五个实验不同的地方：

1. **`Process()` 是两层循环**，外层走 M 块、内层走 $k$；$C$ 在内层循环期间一直留在片上。
2. **`inQueueA` 与 `outQueueC` 的深度是 1**，只有 `inQueueB` 是 2。
3. **`EnQue` 紧跟着 `DeQue`**：对 $A$ 与 $C$ 而言这不是排队，而是取得 MTE2 与矢量流水之间的那一次同步。

内层最核心的两行就是秩一更新。`aLocal.GetValue(i * K_SIZE + k)` 取出的是标量 $A_{ik}$，`bLocal` 是 $B$ 的第 $k$ 行——**它被这段循环复用了 `TM` 次，这就是全部收益的来源**。

In [ ]:
%%writefile -a src_matmul/ascendc_matmul.asc
/* ============ 矩阵乘核函数：秩一更新的矢量实现 ============
 * 模板实参 TM 即复用系数 TILE_M：一次在片上累加 C 的多少行。
 * v1 取 TM = 1（不复用），v2/v3 取 TM = TILE_M。
 */
template <uint32_t TM>
class KernelMatmul {
 public:
  __aicore__ inline KernelMatmul() {}

  __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR c) {
    /* 核间切分沿 M 方向：每个核负责一段连续的行。
     * B 不切分——每个核都要读完整的 B，这正是 6 节所说的
     * 「多核不额外增加 B 的读取总量」的原因：总块数没有变 */
    rowsPerCore_ = M_SIZE / AscendC::GetBlockNum();
    mBlocks_ = rowsPerCore_ / TM;
    const uint32_t rowOffset = AscendC::GetBlockIdx() * rowsPerCore_;

    aGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float *>(a) + rowOffset * K_SIZE,
        rowsPerCore_ * K_SIZE);
    bGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(b), K_SIZE * N_SIZE);
    cGm.SetGlobalBuffer(
        reinterpret_cast<__gm__ float *>(c) + rowOffset * N_SIZE,
        rowsPerCore_ * N_SIZE);

    /* A 与 C 每个 M 块只搬一次，深度取 1；B 逐行流水，深度取 2。见 2.3 节 */
    pipe.InitBuffer(inQueueA, QUEUE_DEPTH_AC, TM * K_SIZE * sizeof(float));
    pipe.InitBuffer(outQueueC, QUEUE_DEPTH_AC, TM * N_SIZE * sizeof(float));
    pipe.InitBuffer(inQueueB, QUEUE_DEPTH_B, N_SIZE * sizeof(float));
    pipe.InitBuffer(tmpBuf, N_SIZE * sizeof(float));
  }

  __aicore__ inline void Process() {
    for (uint32_t mb = 0; mb < mBlocks_; ++mb) {
      /* ---- 搬入本块的 TM 行 A。它们在 GM 上连续，一次 DataCopy 即可 ---- */
      AscendC::LocalTensor<float> aLocal = inQueueA.AllocTensor<float>();
      AscendC::DataCopy(aLocal, aGm[mb * TM * K_SIZE], TM * K_SIZE);
      inQueueA.EnQue(aLocal);
      /* 这一对 EnQue/DeQue 的作用是取得 MTE2 -> 矢量的同步，而不是排队 */
      aLocal = inQueueA.DeQue<float>();

      /* ---- C 的 TM 行驻留片上，先清零 ---- */
      AscendC::LocalTensor<float> cLocal = outQueueC.AllocTensor<float>();
      AscendC::Duplicate(cLocal, 0.0f, TM * N_SIZE);
      AscendC::LocalTensor<float> t = tmpBuf.Get<float>();

      /* ---- 沿 K 做秩一更新 ---- */
      for (uint32_t k = 0; k < K_SIZE; ++k) {
        AscendC::LocalTensor<float> bLocal = inQueueB.AllocTensor<float>();
        AscendC::DataCopy(bLocal, bGm[k * N_SIZE], N_SIZE);
        inQueueB.EnQue(bLocal);
        bLocal = inQueueB.DeQue<float>();

        /* B 的第 k 行被下面这个循环复用 TM 次——收益的全部来源。
         * GetValue 取出标量 A[i][k]，每取一次都要跨一次矢量与标量流水，
         * 整个矩阵乘共取 M x K 次，这是矢量单元做矩阵乘的固有代价（2.2 节） */
        for (uint32_t i = 0; i < TM; ++i) {
          AscendC::Muls(t, bLocal, aLocal.GetValue(i * K_SIZE + k), N_SIZE);
          AscendC::Add(cLocal[i * N_SIZE], cLocal[i * N_SIZE], t, N_SIZE);
        }
        inQueueB.FreeTensor(bLocal);
      }
      inQueueA.FreeTensor(aLocal);

      /* ---- 累加完毕，搬出本块的 TM 行 C ---- */
      outQueueC.EnQue(cLocal);
      cLocal = outQueueC.DeQue<float>();
      AscendC::DataCopy(cGm[mb * TM * N_SIZE], cLocal, TM * N_SIZE);
      outQueueC.FreeTensor(cLocal);
    }
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH_AC> inQueueA;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH_B> inQueueB;
  AscendC::TQue<AscendC::TPosition::VECOUT, QUEUE_DEPTH_AC> outQueueC;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::GlobalTensor<float> aGm, bGm, cGm;
  uint32_t rowsPerCore_ = 0;
  uint32_t mBlocks_ = 0;
};

### 7.3 两个核函数入口

只需要两个入口：一个用 `TILE_M_V1 = 1` 实例化（v1），一个用 `TILE_M` 实例化（v2 与 v3 共用）。**v2 与 v3 调用的是同一个核函数，差别只在启动时给的核数。**

In [ ]:
%%writefile -a src_matmul/ascendc_matmul.asc
/* ===================== 核函数入口 ===================== */

extern "C" __global__ __aicore__ void matmul_novreuse(GM_ADDR a, GM_ADDR b,
                                                      GM_ADDR c) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); /* 声明为纯矢量内核 */
  KernelMatmul<TILE_M_V1> op;
  op.Init(a, b, c);
  op.Process();
}

extern "C" __global__ __aicore__ void matmul_reuse(GM_ADDR a, GM_ADDR b,
                                                   GM_ADDR c) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelMatmul<TILE_M> op; /* 与上面的唯一差别：模板实参 */
  op.Init(a, b, c);
  op.Process();
}

## 8. Host 侧实现

### 8.1 工具函数

ACL 返回值检查、计时与数据生成的写法与前四个实验一致，此处不再复述。本节有三处是本实验特有的：

- **两份 CPU 基准**：`RunCpuNaive` 是朴素 $i$-$j$-$k$，`RunCpuBlocked` 是分块的 $i$-$k$-$j$。后者是加速比的基准。
- **真值用 double 计算**，采用与 `RunCpuBlocked` 相同的循环次序，只把累加类型换成 `double`。
- **容差在运行期推导**，按 4.3 节的式子由 $K$ 与 $\max|C|$ 算出，不使用任何固定常数。

In [ ]:
%%writefile -a src_matmul/ascendc_matmul.asc
/* ============================================================
 *                       Host 侧代码
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

static void NpuInit() {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));
}

static void NpuFinalize() {
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());
}

/* ---------- 数据生成：线性同余发生器，输出 [-1, 1] ---------- */
static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u;
  return static_cast<float>(state >> 8) * (2.0f / 16777216.0f) - 1.0f;
}

static void GenerateInput(std::vector<float> &a, uint32_t seed) {
  uint32_t s = seed;
  for (size_t i = 0; i < a.size(); ++i) {
    a[i] = LcgNextFloat(s);
  }
}

/* ---------- 真值：与分块基准同一循环次序，累加类型换成 double ---------- */
static void GoldenBlocked(const std::vector<float> &a,
                          const std::vector<float> &b, std::vector<double> &c) {
  std::fill(c.begin(), c.end(), 0.0);
  for (uint32_t ii = 0; ii < M_SIZE; ii += CPU_BM) {
    const uint32_t iEnd = (ii + CPU_BM < M_SIZE) ? ii + CPU_BM : M_SIZE;
    for (uint32_t kk = 0; kk < K_SIZE; kk += CPU_BK) {
      const uint32_t kEnd = (kk + CPU_BK < K_SIZE) ? kk + CPU_BK : K_SIZE;
      for (uint32_t i = ii; i < iEnd; ++i) {
        for (uint32_t k = kk; k < kEnd; ++k) {
          const double av = static_cast<double>(a[i * K_SIZE + k]);
          for (uint32_t j = 0; j < N_SIZE; ++j) {
            c[i * N_SIZE + j] += av * static_cast<double>(b[k * N_SIZE + j]);
          }
        }
      }
    }
  }
}

/* ---------- CPU 基准一：朴素 i-j-k ----------
 * 内层 k 变化时 B 按列访问，跨步 N，局部性最差。这是「反面教材」，
 * 不参与加速比，只用于回答「仅仅换一下循环次序能差多少」 */
static double RunCpuNaive(const std::vector<float> &a,
                          const std::vector<float> &b, std::vector<float> &c) {
  const double t0 = GetTimeMs();
  for (uint32_t i = 0; i < M_SIZE; ++i) {
    for (uint32_t j = 0; j < N_SIZE; ++j) {
      float s = 0.0f;
      for (uint32_t k = 0; k < K_SIZE; ++k) {
        s += a[i * K_SIZE + k] * b[k * N_SIZE + j];
      }
      c[i * N_SIZE + j] = s;
    }
  }
  return GetTimeMs() - t0;
}

/* ---------- CPU 基准二：分块 + i-k-j ----------
 * 与第五章 OpenMP GEMM 的做法一致（此处不开多线程）。
 * 内层 j 连续访问 B 与 C，编译器可以直接向量化。加速比以本版为基准 */
static double RunCpuBlocked(const std::vector<float> &a,
                            const std::vector<float> &b, std::vector<float> &c,
                            int repeat) {
  double best = 0.0;
  for (int r = 0; r < repeat; ++r) {
    std::fill(c.begin(), c.end(), 0.0f);
    const double t0 = GetTimeMs();
    for (uint32_t ii = 0; ii < M_SIZE; ii += CPU_BM) {
      const uint32_t iEnd = (ii + CPU_BM < M_SIZE) ? ii + CPU_BM : M_SIZE;
      for (uint32_t kk = 0; kk < K_SIZE; kk += CPU_BK) {
        const uint32_t kEnd = (kk + CPU_BK < K_SIZE) ? kk + CPU_BK : K_SIZE;
        for (uint32_t jj = 0; jj < N_SIZE; jj += CPU_BN) {
          const uint32_t jEnd = (jj + CPU_BN < N_SIZE) ? jj + CPU_BN : N_SIZE;
          for (uint32_t i = ii; i < iEnd; ++i) {
            for (uint32_t k = kk; k < kEnd; ++k) {
              const float av = a[i * K_SIZE + k];
              for (uint32_t j = jj; j < jEnd; ++j) {
                c[i * N_SIZE + j] += av * b[k * N_SIZE + j];
              }
            }
          }
        }
      }
    }
    const double dt = GetTimeMs() - t0;
    if (r == 0 || dt < best) best = dt;
  }
  return best;
}

/* ---------- 校验：绝对误差判定，理由见 4.4 节 ---------- */
static bool Verify(const char *ver, const std::vector<float> &out,
                   const std::vector<double> &golden, double atol) {
  double maxAbs = 0.0;
  size_t badIdx = 0;
  for (size_t i = 0; i < golden.size(); ++i) {
    const double e = std::fabs(static_cast<double>(out[i]) - golden[i]);
    if (e > maxAbs) {
      maxAbs = e;
      badIdx = i;
    }
  }
  const bool ok = (maxAbs <= atol);
  std::printf(
      "[VERIFY] ver=%s atol=%.3e max_abs_err=%.3e margin=%.2f at=%llu "
      "result=%s\n",
      ver, atol, maxAbs, atol / (maxAbs > 0.0 ? maxAbs : atol),
      static_cast<unsigned long long>(badIdx), ok ? "PASS" : "FAIL");
  return ok;
}

/* ---------- 打印一行可被程序解析的性能记录 ----------
 * tile_m     复用系数
 * b_read_gb  B 的读取总量，按 (M/TM) * K * N * 4 核算
 * intensity  计算访存比 = TM / 2 FLOP/Byte
 * ub_kb      单核片上占用 = 8*TM + 12 KB */
static void ReportPerf(const char *ver, uint32_t blockDim, uint32_t tm,
                       double cpuMs, double kernelMs) {
  const double gflop = 2.0 * M_SIZE * N_SIZE * K_SIZE / 1e9;
  const double bReadGB =
      static_cast<double>(M_SIZE / tm) * K_SIZE * N_SIZE * sizeof(float) / 1e9;
  std::printf(
      "[PERF]   ver=%s m=%u n=%u k=%u blockDim=%u tile_m=%u ub_kb=%.1f "
      "b_read_gb=%.3f intensity=%.2f cpu_ms=%.4f kernel_ms=%.4f "
      "gflops=%.2f sp_kernel=%.4f\n",
      ver, M_SIZE, N_SIZE, K_SIZE, blockDim, tm, 8.0 * tm + 12.0, bReadGB,
      tm / 2.0, cpuMs, kernelMs, gflop / (kernelMs / 1000.0),
      cpuMs / kernelMs);
}

/* ---------- 计时宏：纯核函数耗时（不含 H2D/D2H） ---------- */
#define TIME_KERNEL(LAUNCH, OUT_MS)                                               \
  do {                                                                            \
    for (int _w = 0; _w < WARMUP; ++_w) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream));                                \
    }                                                                             \
    const double _t0 = GetTimeMs();                                               \
    for (int _r = 0; _r < REPEAT; ++_r) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream)); /* 先同步再停止计时 */ \
    }                                                                             \
    (OUT_MS) = (GetTimeMs() - _t0) / REPEAT;                                      \
  } while (0)

### 8.2 主程序

结构与前四个实验一致。$A$、$B$ 在三个版本之间保持常驻设备，只在最后把 $C$ 搬回主机做校验；主机与设备之间的传输不计入耗时，理由见第 4.1 节。

启动前先检查规模是否可整除：v3 要求 $M$ 能被 $\text{blockDim} \times \text{TILE\_M}$ 整除。

In [ ]:
%%writefile -a src_matmul/ascendc_matmul.asc
/* ---------- 三个版本的下发。写成函数的理由见 8.2 节 ---------- */
static void LaunchV1(uint8_t *a, uint8_t *b, uint8_t *c) {
  matmul_novreuse<<<1, nullptr, g_stream>>>(a, b, c); /* 单核，TILE_M = 1 */
}

static void LaunchV2(uint8_t *a, uint8_t *b, uint8_t *c) {
  matmul_reuse<<<1, nullptr, g_stream>>>(a, b, c); /* 单核，TILE_M = 16 */
}

static void LaunchV3(uint8_t *a, uint8_t *b, uint8_t *c) {
  matmul_reuse<<<BLOCK_DIM, nullptr, g_stream>>>(a, b,
                                                 c); /* 与 v2 同一核函数 */
}

/* 每个版本的固定流程：运行一次并校验 -> 计时 -> 报告。
 * A 与 B 在三个版本之间保持常驻设备，主机与设备之间的传输不计入耗时：
 * 判断卸载是否划算的单位是整段计算图，而不是单个算子，见 4.1 节 */
#define RUN_VERSION(TAG, LAUNCH, BD, TM)                      \
  do {                                                        \
    LAUNCH;                                                   \
    ACL_CHECK(aclrtSynchronizeStream(g_stream));              \
    ACL_CHECK(aclrtMemcpy(hostC.data(), bytesC, cDev, bytesC, \
                          ACL_MEMCPY_DEVICE_TO_HOST));        \
    allPass &= Verify(TAG, hostC, golden, atol);              \
    TIME_KERNEL(LAUNCH, kMs);                                 \
    ReportPerf(TAG, BD, TM, cpuMs, kMs);                      \
  } while (0)

int32_t main() {
  /* ---------- 规模合法性检查 ---------- */
  /* 先查片上占用：TILE_M 超界时这条信息比整除性更有价值（动手练习第 1 题） */
  if (8 * TILE_M + 12 > 192) {
    std::printf("[FATAL] TILE_M = %u 时片上占用 %u KB，超出 UB 的 192 KB\n",
                TILE_M, 8 * TILE_M + 12);
    return 1;
  }
  if (M_SIZE % (BLOCK_DIM * TILE_M) != 0) {
    std::printf("[FATAL] M 必须能被 blockDim x TILE_M = %u 整除，当前 M = %u\n",
                BLOCK_DIM * TILE_M, M_SIZE);
    return 1;
  }

  const size_t nA = static_cast<size_t>(M_SIZE) * K_SIZE;
  const size_t nB = static_cast<size_t>(K_SIZE) * N_SIZE;
  const size_t nC = static_cast<size_t>(M_SIZE) * N_SIZE;

  std::vector<float> hostA(nA), hostB(nB), hostC(nC), cpuOut(nC);
  std::vector<double> golden(nC);
  GenerateInput(hostA, 2026u);
  GenerateInput(hostB, 20260521u);

  /* ---------- 真值与容差 ---------- */
  GoldenBlocked(hostA, hostB, golden);
  double maxAbsC = 0.0;
  for (size_t i = 0; i < nC; ++i) {
    const double v = std::fabs(golden[i]);
    if (v > maxAbsC) maxAbsC = v;
  }
  /* 4.3 节的式子：atol = c * eps * sqrt(K) * max|C| */
  const double atol =
      ATOL_COEF * F32_EPS * std::sqrt(static_cast<double>(K_SIZE)) * maxAbsC;

  /* ---------- 两份 CPU 基准 ---------- */
  const double cpuMs = RunCpuBlocked(hostA, hostB, cpuOut, 3);
  double cpuBlockedErr = 0.0;
  for (size_t i = 0; i < nC; ++i) {
    const double e = std::fabs(static_cast<double>(cpuOut[i]) - golden[i]);
    if (e > cpuBlockedErr) cpuBlockedErr = e;
  }
  double cpuNaiveMs = 0.0;
#if (LAB_SKIP_CPU_NAIVE == 0)
  cpuNaiveMs = RunCpuNaive(hostA, hostB, cpuOut);
#endif

  const double gflop = 2.0 * M_SIZE * N_SIZE * K_SIZE / 1e9;
  std::printf(
      "M=%u  N=%u  K=%u  TILE_M=%u  BLOCK_DIM=%u  REPEAT=%d  总运算量=%.3f "
      "GFLOP\n",
      M_SIZE, N_SIZE, K_SIZE, TILE_M, BLOCK_DIM, REPEAT, gflop);
  std::printf(
      "[BASE]   cpu_blocked_ms=%.2f cpu_naive_ms=%.2f cpu_blocked_gflops=%.2f "
      "max_abs_c=%.4f atol=%.3e cpu_blocked_max_abs_err=%.3e\n",
      cpuMs, cpuNaiveMs, gflop / (cpuMs / 1000.0), maxAbsC, atol,
      cpuBlockedErr);

  /* ---------- 申请显存并搬入 ---------- */
  NpuInit();
  const size_t bytesA = nA * sizeof(float);
  const size_t bytesB = nB * sizeof(float);
  const size_t bytesC = nC * sizeof(float);
  uint8_t *aDev = nullptr, *bDev = nullptr, *cDev = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&aDev, bytesA, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&bDev, bytesB, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&cDev, bytesC, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(aDev, bytesA, hostA.data(), bytesA,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(aclrtMemcpy(bDev, bytesB, hostB.data(), bytesB,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  bool allPass = true;
  double kMs = 0.0;

  RUN_VERSION("v1", LaunchV1(aDev, bDev, cDev), 1, TILE_M_V1);
  RUN_VERSION("v2", LaunchV2(aDev, bDev, cDev), 1, TILE_M);
  RUN_VERSION("v3", LaunchV3(aDev, bDev, cDev), BLOCK_DIM, TILE_M);

  ACL_CHECK(aclrtFree(cDev));
  ACL_CHECK(aclrtFree(bDev));
  ACL_CHECK(aclrtFree(aDev));
  NpuFinalize();

  std::printf(allPass ? "[SUCCESS] 全部版本通过校验。\n"
                      : "[FAILED] 存在未通过校验的版本！\n");
  return allPass ? 0 : 1;
}

## 9. 编译与运行

`-O2` 不能省略：两份 CPU 基准与 NPU 代码在同一次编译中生成，用 `-O0` 会人为放大基准的耗时，使加速比失真。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_matmul/ascendc_matmul.asc"
EXE = "src_matmul/ascendc_matmul"

# 编译选项集中定义一处，§11 与 §12 的参数扫描直接复用
FLAGS = ["--npu-arch=" + ARCH, "-O2"]

# bisheng [算子源文件] [编译选项] -o [输出产物名称]
cmd = ["bisheng", SRC] + FLAGS + ["-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

程序会先算真值、再跑两份 CPU 基准，然后才轮到 NPU。**朴素 CPU 基准在 $1024^3$ 规模下需要数秒，请耐心等待。**

In [ ]:
def run_demo(args=(), exe=None, timeout=1800):
    # 与实验二至实验五同名同约定：只返回 stdout
    proc = subprocess.run(
        ["./" + (exe or EXE)] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print(out_main)

### 9.1 解析输出

In [ ]:
def parse_rows(text, tag):
    # 把所有以 [tag] 开头的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[" + tag + "]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k in ("ver", "result") else float(v)
            rows.append(d)
    return rows


def parse_perf(text):
    return parse_rows(text, "PERF")


def parse_verify(text):
    return {r["ver"]: r for r in parse_rows(text, "VERIFY")}


def parse_base(text):
    rows = parse_rows(text, "BASE")
    return rows[0] if rows else None


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
base_main = parse_base(out_main)
base_v1 = rows_main[0]["kernel_ms"] if rows_main else 1.0
GFLOP = 2.0 * 1024**3 / 1e9

print(
    "CPU 分块基准：%.1f ms（%.2f GFLOPS）    CPU 朴素基准：%.1f ms    朴素/分块 = %.2fx"
    % (
        base_main["cpu_blocked_ms"],
        base_main["cpu_blocked_gflops"],
        base_main["cpu_naive_ms"],
        base_main["cpu_naive_ms"] / base_main["cpu_blocked_ms"],
    )
)
print(
    "容差 atol = 2·ε·√K·max|C| = %.3e   （max|C| = %.3f）    CPU float32 自身误差 %.3e"
    % (base_main["atol"], base_main["max_abs_c"], base_main["cpu_blocked_max_abs_err"])
)
print()

hdr = (
    "版本",
    "TILE_M",
    "核数",
    "UB(KB)",
    "B读取(GB)",
    "访存比",
    "kernel(ms)",
    "GFLOPS",
    "vs CPU",
    "vs v1",
    "判定",
)
print("%-5s %7s %5s %8s %10s %7s %11s %9s %8s %8s %6s" % hdr)
print("-" * 98)
for r in rows_main:
    print(
        "%-5s %7d %5d %8.1f %10.3f %7.1f %11.4f %9.2f %7.1fx %7.2fx %6s"
        % (
            r["ver"],
            r["tile_m"],
            r["blockDim"],
            r["ub_kb"],
            r["b_read_gb"],
            r["intensity"],
            r["kernel_ms"],
            r["gflops"],
            r["sp_kernel"],
            base_v1 / r["kernel_ms"],
            chk_main[r["ver"]]["result"],
        )
    )

## 10. 结果可视化

左图是三个版本相对 CPU 分块基准的加速比，右图是达到的 GFLOPS。口径只计核函数耗时。

**请重点看右图。** 加速比只说明比这台机器上的 CPU 快多少，换一台主机数值就会变；GFLOPS 是绝对吞吐，可以跨机器比较，也可以与第 12 节的 Cube 版本直接放在一起。


In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_KERNEL, C_E2E, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
labels = [
    "%s\nTILE_M=%d, %d core" % (r["ver"], r["tile_m"], r["blockDim"]) for r in rows_main
]
xpos = np.arange(len(vers))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), dpi=120)

ax = axes[0]
ax.bar(
    xpos,
    [r["sp_kernel"] for r in rows_main],
    0.46,
    color=C_KERNEL,
    label="NPU kernel time",
)
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.text(
    -0.45, 1.1, "baseline: CPU 1 thread, blocked = 1.0x", fontsize=8.5, color="#777777"
)
for i, r in enumerate(rows_main):
    ax.text(
        i,
        r["sp_kernel"],
        "%.1fx" % r["sp_kernel"],
        ha="center",
        va="bottom",
        fontsize=8.5,
    )
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=8.5)
ax.set_yscale("log")
ax.set_ylabel("Speedup over CPU baseline")
ax.set_title("Lab 6: speedup vs single-thread blocked CPU")
ax.grid(axis="y", alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left", fontsize=9)

ax = axes[1]
gf = [r["gflops"] for r in rows_main]
ax.bar(xpos, gf, 0.5, color=[C_BASE, C_KERNEL, C_OPT])
ax.axhline(base_main["cpu_blocked_gflops"], color=C_E2E, lw=1.2, ls="--")
ax.text(
    len(vers) - 0.5,
    base_main["cpu_blocked_gflops"] * 1.05,
    "CPU blocked  %.1f GFLOPS" % base_main["cpu_blocked_gflops"],
    fontsize=8.5,
    color=C_E2E,
    ha="right",
)
for i, v in enumerate(gf):
    ax.text(i, v, "%.0f" % v, ha="center", va="bottom", fontsize=9)
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel("Achieved GFLOPS (fp32, vector unit)")
ax.set_title("Lab 6: achieved throughput")
ax.grid(axis="y", alpha=0.3)

for a in axes:
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 11. 复用系数扫描：收益在哪一档消失

v1 与 v2 只差一个复用系数，中间还有几档没有测过。以 `-DLAB_TILE_M` 依次取 1、2、4、8、16，**固定单核**重新编译运行，看耗时随复用系数如何变化。

每一档的搬运量按 $1/T$ 下降，计算访存比按 $T/2$ 上升，而运算量一条指令都没有变。因此这条曲线要回答的问题很具体：**搬运量降到什么程度之后，耗时就不再跟着降了？**

扫描期间用 `-DLAB_SKIP_CPU_NAIVE=1` 跳过耗时数秒的朴素 CPU 基准，并把 `LAB_REPEAT` 降到 3。


In [ ]:
_built = {}


def build_and_run(defines=None, args=(), tag="scan", timeout=1800):
    # 与实验二至实验五同名同约定；同一个 tag 只编译一次
    exe = "src_matmul/ascendc_matmul_%s" % tag
    if tag not in _built:
        cmd = ["bisheng", SRC] + FLAGS + ["-o", exe]  # 与 §9 使用同一组编译选项
        for k, v in (defines or {}).items():
            cmd.append("-D%s=%s" % (k, v))
        b = subprocess.run(cmd, capture_output=True, text=True)
        _built[tag] = b.returncode == 0
        if not _built[tag]:
            print("❌ 编译失败（%s）：%s" % (tag, (b.stdout + b.stderr).strip()[-300:]))
    if not _built[tag]:
        return "", False
    r = subprocess.run(
        ["./" + exe] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    # 程序在「有版本未通过校验」时返回 1，这不是运行失败。
    # 判断是否正常运行要看有没有产出 [PERF] 记录行，而不是看返回码
    return r.stdout, ("[PERF]" in r.stdout)


COMMON = {"LAB_SKIP_CPU_NAIVE": 1, "LAB_REPEAT": 3}
tile_ms = [1, 2, 4, 8, 16]
scan_tile = []

for tm in tile_ms:
    d = dict(COMMON)
    d["LAB_TILE_M"] = tm
    d["LAB_BLOCK_DIM"] = 1  # 固定单核，只让复用系数变化
    txt, ok = build_and_run(d, tag="tm%d" % tm)
    if not ok:
        print("❌ TILE_M=%-3d 编译或运行失败" % tm)
        continue
    p = {r["ver"]: r for r in parse_perf(txt)}["v2"]
    scan_tile.append(p)
    print(
        "TILE_M=%-3d 强度=%5.1f FLOP/B  UB=%5.1f KB  B读取=%6.3f GB  "
        "kernel=%8.3f ms  %7.2f GFLOPS  等效带宽=%6.1f GB/s"
        % (
            tm,
            p["intensity"],
            p["ub_kb"],
            p["b_read_gb"],
            p["kernel_ms"],
            p["gflops"],
            p["b_read_gb"] / (p["kernel_ms"] / 1000.0),
        )
    )

In [ ]:
if len(scan_tile) >= 3:
    tms = np.array([p["tile_m"] for p in scan_tile])
    ms = np.array([p["kernel_ms"] for p in scan_tile])
    gfl = np.array([p["gflops"] for p in scan_tile])
    gb = np.array([p["b_read_gb"] for p in scan_tile])

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2), dpi=120)

    # 左图：搬运量按 1/T 下降，耗时并没有一直跟着下降
    ax = axes[0]
    ax.plot(tms, gb, marker="o", lw=2, color=C_BASE, label="B read volume (GB)")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(tms)
    ax.set_xticklabels(tms)
    ax.set_xlabel("TILE_M (reuse factor)")
    ax.set_ylabel("B read volume (GB)", color=C_BASE)
    ax.tick_params(axis="y", labelcolor=C_BASE)
    ax.grid(alpha=0.3, which="both")

    ax2 = ax.twinx()
    ax2.plot(tms, ms, marker="s", lw=2, color=C_KERNEL)
    ax2.set_yscale("log")
    ax2.set_ylabel("kernel time (ms)", color=C_KERNEL)
    ax2.tick_params(axis="y", labelcolor=C_KERNEL)
    ax.set_title("Lab 6: traffic falls as 1/T, time does not")

    # 右图：达到的 GFLOPS 随复用系数饱和
    ax = axes[1]
    ax.plot(tms, gfl, marker="s", lw=2, color=C_OPT)
    for x, y in zip(tms, gfl):
        ax.annotate("%.1f" % y, (x, y), textcoords="offset points",
                    xytext=(6, -11), fontsize=8.5, color=C_OPT)
    ax.axhline(gfl.max(), lw=1.0, ls=":", color="#888888")
    ax.text(tms[0], gfl.max() * 1.03,
            " single-core vector ceiling reached here", fontsize=8.5, color="#777777")
    ax.set_xscale("log", base=2)
    ax.set_xticks(tms)
    ax.set_xticklabels(tms)
    ax.set_xlabel("TILE_M (reuse factor)")
    ax.set_ylabel("Achieved GFLOPS (1 core)")
    ax.set_title("Lab 6: reuse gain saturates")
    ax.grid(alpha=0.3)

    for a in axes:
        for s in ("top", "right"):
            a.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

    print("复用系数由 %d 增到 %d：B 的读取总量降为 1/%d，而耗时只降到 %.2f 倍。"
          % (tms[0], tms[-1], tms[-1] // tms[0], ms[-1] / ms[0]))
    print("单核达到的最高吞吐约 %.2f GFLOPS。" % gfl.max())

    # 找出收益消失的那一档：从这一档起，再增大复用系数带来的改善不足 1%
    knee = tms[-1]
    for a, b, t in zip(ms[:-1], ms[1:], tms[:-1]):
        if (a - b) / a < 0.01:
            knee = t
            break
    print("收益在 TILE_M = %d 附近消失：此后耗时的变化已不足 1%%，"
          "而片上容量允许的上界是 22。" % knee)


## 12. 用 Cube 单元重做同一个矩阵乘

前面三个版本已经把矢量单元这条实现路径的余量用尽：复用系数再往上增大不再有收益，可调的只剩核数。本节改用另一条路径——**把同一个矩阵乘交给专门为它设计的 Cube 单元**，用 CANN 主推的 `Matmul` 高阶 API 实现。

### 12.1 两条不同的片上通路

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
|  | 矢量单元（AIV）路径 | Cube 单元（AIC）路径 |
| --- | --- | --- |
| 数据经过的片上存储 | Unified Buffer（UB） | L1 → L0A / L0B，结果落在 L0C |
| 标量是否参与 | 参与：每次乘加前要取一个标量 | 不参与 |
| 一条指令完成多少次乘加 | 一对 <code>Muls</code>+<code>Add</code> 处理 $N$ 个元素 | 一个 $16\times16\times16$ 的块 |
| 数据格式 | ND（行优先） | L1 中转换为分形格式 |
| 结果写回 | MTE3 | FixPipe |
| 分块由谁决定 | 开发者写在代码里（<code>TILE_M</code>） | Tiling 计算得到，API 内部使用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">矢量单元（AIV）路径</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Cube 单元（AIC）路径</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据经过的片上存储</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Unified Buffer（UB）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">L1 → L0A / L0B，结果落在 L0C</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标量是否参与</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参与：每次乘加前要取一个标量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不参与</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一条指令完成多少次乘加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一对 <code>Muls</code>+<code>Add</code> 处理 $N$ 个元素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一个 $16\times16\times16$ 的块</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据格式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ND（行优先）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">L1 中转换为分形格式</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">结果写回</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">MTE3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">FixPipe</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分块由谁决定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">开发者写在代码里（<code>TILE_M</code>）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Tiling 计算得到，API 内部使用</td>
</tr>
</tbody>
</table>

矢量单元这条通路上，数据经 UB 进出，标量还要绕经标量单元。Cube 的通路完全不同：$A$、$B$ 从 Global Memory 经 **L1**（在这里完成 ND 到分形格式的转换）进入 **L0A / L0B**，Cube 算出的结果落在 **L0C**（以 float32 累加），最后经 **FixPipe** 写回 Global Memory。**整条通路不经过 UB，也没有标量参与。**

因此第 3 节那套 UB 占用核算对 Cube 并不适用：两者用的不是同一块片上存储。第 13 节 ② 那三项限制，Cube 逐条消除了其中两项。

<img src="./images/06.06_cube_path.png" alt="06.06_cube_path"  width="900px" >

官方的硬件架构图把这两条通路画在了同一张图里。上半部分是 AIC，下半部分是 AIV：**AIC 一侧有 L1、L0A / L0B、L0C，没有 Unified Buffer；AIV 一侧有 Unified Buffer，没有 L0。**两侧各有自己的 Scalar 与指令序列，通过 L2 Cache 连到 Global Memory。

<img src="./images/06.06_ai_core_220x.png" alt="06.06_ai_core_220x" width="900px">

*NPU 架构版本 220x 的硬件架构图。220x 对应的产品型号即 Atlas A2 / A3 训练推理系列产品*

### 12.2 一条指令的规模

矢量单元上，一对 `Muls` + `Add` 处理 $N$ 个元素、完成 $N$ 次乘加，用掉**两条**指令。Cube 不同——官方在讲基本块选择时给出了这个口径：

> 在输入为 fp16 类型的情况下，Cube 执行单元 1 cycle 能算 $16 \times 16 \times 16$ 个数。
> ——《Ascend C 算子开发指南》「Matmul 基本块选择」一节

即一个 $16^3 = 4096$ 次乘加的块。这是硬件分工带来的差别，与实现的优劣无关。

### 12.3 `Matmul` 高阶 API 的五个步骤

官方把 Kernel 侧的用法归纳为五步，本节的核函数就是这五步的直接落地：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 调用 | 作用 |
| --- | --- | --- |
| 1 创建对象 | <code>Matmul&lt;A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE&gt; mm;</code> | 四个类型参数各自给出内存逻辑位置、数据格式与数据类型 |
| 2 初始化 | <code>REGIST_MATMUL_OBJ(&amp;pipe, GetSysWorkSpacePtr(), mm, &amp;tiling);</code> | 绑定 <code>TPipe</code>、系统 workspace 与切分参数 |
| 3 设置输入 | <code>mm.SetTensorA(...); mm.SetTensorB(...);</code> | 给出本核负责的那一块 $A$、$B$ 的起始地址 |
| 4 完成计算 | <code>mm.IterateAll(gmC);</code> | 一次算完本核负责的全部数据；另有 <code>Iterate</code> 逐次迭代的写法 |
| 5 结束 | <code>mm.End();</code> | 释放 API 内部占用的资源 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">调用</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 创建对象</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Matmul&lt;A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE&gt; mm;</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四个类型参数各自给出内存逻辑位置、数据格式与数据类型</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2 初始化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>REGIST_MATMUL_OBJ(&amp;pipe, GetSysWorkSpacePtr(), mm, &amp;tiling);</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">绑定 <code>TPipe</code>、系统 workspace 与切分参数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3 设置输入</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>mm.SetTensorA(...); mm.SetTensorB(...);</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">给出本核负责的那一块 $A$、$B$ 的起始地址</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4 完成计算</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>mm.IterateAll(gmC);</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一次算完本核负责的全部数据；另有 <code>Iterate</code> 逐次迭代的写法</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">5 结束</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>mm.End();</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">释放 API 内部占用的资源</td>
</tr>
</tbody>
</table>

与第 7.2 节的两层循环相比，分块、L1 与 L0 之间的搬运、ND 与分形格式之间的转换，都不再出现在核函数里。

官方把主机侧与核函数侧的分工画成了一张流程图。左侧是主机侧的 Tiling 生成，右侧是核函数侧的五步，中间那条箭头就是 12.4 节要解决的问题——**Tiling 参数怎么从主机侧交到核函数侧**：

<img src="./images/06.06_matmul_flow.png" alt="06.06_matmul_flow" width="880px">

*矩阵编程流程示意图。图中「完成矩阵乘操作」一步给出两种写法：`Iterate` 配 `while` 循环逐次迭代，或 `IterateAll` 一次算完；本节用后者*

### 12.4 Tiling 从哪里来

代价是切分参数不再是编译期常量。`Matmul` 需要一个 `TCubeTiling` 结构体，由 CANN 的 Tiling 库根据**平台信息**与**矩阵形状**算出：主机侧调用 `MultiCoreMatmulTiling`，把结果拷到设备内存，核函数再从那里读回来。

这正是第 7 节与本节最大的工程差别：**矢量版的 `TILE_M` 是写死在源码里的模板实参，Cube 版的切分参数是运行期由库算出来的。** 前者换一个形状就要重编译，后者不必。

`Matmul` 高阶 API 内部还需要一块**系统 workspace**。官方对直调工程给出的做法是开启编译宏：

> Kernel 直调算子开发场景需要使用 workspace 空间时，建议开启编译选项 HAVE_WORKSPACE。host 侧开发者仍需要自行申请 workspace 的空间，并传入。……开启 HAVE_WORKSPACE 后，开发者在 kernel 侧入参处获取的 workspace 为偏移了系统 workspace 后的用户 workspace。
> ——《Ascend C 算子开发指南》 Kernel 直调工程的 workspace 说明

配套还要开启 `HAVE_TILING`：开启之后，框架把核函数入参的**最后一个**当作 tiling、**倒数第二个**当作 workspace——这正是本节核函数 `(a, b, c, workspace, tilingGm)` 的排布。两个宏都开启之后，核函数里不必再写任何 workspace 相关的代码。

> **一处接口变更**：在不开启这两个宏的旧写法里，需要在初始化前手工调用 `SetSysWorkspace`。该接口已被官方标记为废弃，编译时会给出告警。本节因此走编译宏这条路；源码中仍保留了一个 `#ifndef HAVE_WORKSPACE` 分支，万一编译宏在你的环境上不生效，去掉这两个 `-D` 选项即可回到旧写法。

### 12.5 数据类型与精度口径

本节的 $A$、$B$ 取 **half**，$C$ 取 **float32**——这是官方样例覆盖最广的组合。官方另有一条硬件说明：

> Cube 计算场景 float 算力为 half 算力的 1/4。
> ——《Ascend C 算子开发指南》硬件约束

**因此 v4 与前三个版本的误差不能直接比较**：v1–v3 的输入是 float32，误差来自沿 $K$ 的累加舍入；v4 的输入被量化到 half，误差主要来自这一步量化，量级要大得多。为了把两件事分开，本节的真值按**量化之后的 $A$、$B$** 以 double 重新计算，容差仍按第 4.3 节的公式推导。这样得到的是 Cube 把它拿到的那份数据算得准不准，而不是 half 相对 float32 损失了多少。


In [ ]:
%%writefile src_matmul/ascendc_matmul_cube.asc
/**
 * 并行计算 第六章 实验六：矩阵乘法 MatMul（Cube 单元 · Matmul 高阶 API）
 *
 * 与 ascendc_matmul.asc 的差别：
 *   - 计算落在 Cube 单元（AIC）上，不使用矢量单元；
 *   - 分块、L1 与 L0 之间的搬运、ND 与分形格式的转换，全部由 Matmul 高阶 API 承担；
 *   - A、B 为 half，C 为 float32——官方样例覆盖最广的组合，见 12.5 节。
 *
 * 编译命令见第 12.7 节：比前面多了 Matmul 高阶 API 所需的几个链接库，
 * 以及 HAVE_WORKSPACE / HAVE_TILING 两个编译宏。
 *
 * 入参顺序必须是（输入…, 输出, workspace, tiling）：开启 HAVE_TILING 后，
 * 框架把最后一个参数当作 tiling、倒数第二个当作 workspace。
 */

/* 纯 Cube 模式。这一行必须出现在 include "lib/matmul_intf.h" 之前 */
#define ASCENDC_CUBE_ONLY

#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "lib/matmul_intf.h"    /* Matmul 高阶 API（Device 侧） */
#include "tiling/tiling_api.h"  /* Tiling 库（Host 侧） */

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */
#ifndef LAB_M
#define LAB_M (1024)
#endif
#ifndef LAB_N
#define LAB_N (1024)
#endif
#ifndef LAB_K
#define LAB_K (1024)
#endif
constexpr uint32_t M_SIZE = static_cast<uint32_t>(LAB_M);
constexpr uint32_t N_SIZE = static_cast<uint32_t>(LAB_N);
constexpr uint32_t K_SIZE = static_cast<uint32_t>(LAB_K);

#ifndef LAB_REPEAT
#define LAB_REPEAT (10)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 2;

constexpr double F32_EPS = 1.1920928955078125e-7;
constexpr double ATOL_COEF = 2.0;

/* ============================================================
 *                       Device 侧
 * ============================================================ */

/* 步骤 1 的四个类型参数：内存逻辑位置、数据格式、数据类型 */
using A_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>;
using B_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>;
using C_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, float>;
using BIAS_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, float>;

/* 把主机侧算好的 TCubeTiling 从 Global Memory 读进核函数的局部变量。
 * TCubeTiling 是一个纯数据结构，按 4 字节为单位逐字复制即可 */
__aicore__ inline void CopyTiling(TCubeTiling *tiling, GM_ADDR tilingGm) {
  uint32_t *dst = reinterpret_cast<uint32_t *>(tiling);
  __gm__ uint32_t *src = reinterpret_cast<__gm__ uint32_t *>(tilingGm);
  for (uint32_t i = 0; i < sizeof(TCubeTiling) / sizeof(uint32_t); ++i) {
    dst[i] = src[i];
  }
}

/* 本核负责 C 的哪一块。Tiling 已经给出单核处理的形状 singleCoreM / singleCoreN，
 * 这里只需按 blockIdx 换算成三个矩阵上的偏移，并处理最后一块可能不满的情形 */
__aicore__ inline void CalcGMOffset(int blockIdx, const TCubeTiling &tiling,
                                    int &offsetA, int &offsetB, int &offsetC,
                                    int &tailM, int &tailN) {
  uint32_t mSingleBlocks =
      (tiling.M + tiling.singleCoreM - 1) / tiling.singleCoreM;
  uint32_t mCoreIndx = blockIdx % mSingleBlocks;
  uint32_t nCoreIndx = blockIdx / mSingleBlocks;

  offsetA = mCoreIndx * tiling.Ka * tiling.singleCoreM;
  offsetB = nCoreIndx * tiling.singleCoreN;
  offsetC = mCoreIndx * tiling.N * tiling.singleCoreM +
            nCoreIndx * tiling.singleCoreN;

  tailM = tiling.M - mCoreIndx * tiling.singleCoreM;
  tailM = tailM < tiling.singleCoreM ? tailM : tiling.singleCoreM;
  tailN = tiling.N - nCoreIndx * tiling.singleCoreN;
  tailN = tailN < tiling.singleCoreN ? tailN : tiling.singleCoreN;
}

extern "C" __global__ __aicore__ void matmul_cube(GM_ADDR a, GM_ADDR b,
                                                  GM_ADDR c, GM_ADDR workspace,
                                                  GM_ADDR tilingGm) {
  /* 声明 Kernel 类型。官方文档中没有「纯 Cube」这一取值，
   * 可用的最接近取值是 AIC : AIV = 1 : 1：AIV 侧照常被启动，
   * 但紧接着的一行会让它立即返回 */
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_1);

  /* 纯 Cube 模式：AIV 侧不做任何事，直接返回 */
  if (g_coreType == AscendC::AIV) {
    return;
  }

  AscendC::TPipe pipe;
  TCubeTiling tiling;
  CopyTiling(&tiling, tilingGm);

  AscendC::GlobalTensor<half> aGlobal, bGlobal;
  AscendC::GlobalTensor<float> cGlobal;
  aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(a),
                          tiling.M * tiling.Ka);
  bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(b),
                          tiling.Kb * tiling.N);
  cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(c),
                          tiling.M * tiling.N);

  int offsetA = 0, offsetB = 0, offsetC = 0, tailM = 0, tailN = 0;
  CalcGMOffset(AscendC::GetBlockIdx(), tiling, offsetA, offsetB, offsetC, tailM,
               tailN);

  /* ---- 步骤 1：创建 Matmul 对象 ---- */
  AscendC::Matmul<A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE> mm;

  /* 开启 HAVE_WORKSPACE 编译宏之后，框架会自动把入参倒数第二个参数识别为 workspace，
   * 并在核函数侧设置好系统 workspace，这里不需要再做任何事。见 12.4 节。
   * 下面这段只在未开启该宏时才编译进去，作为回退路径保留：
   * AscendC::SetSysWorkspace 已被官方标记为废弃，会产生编译告警 */
#ifndef HAVE_WORKSPACE
  AscendC::SetSysWorkspace(workspace);
  if (GetSysWorkSpacePtr() == nullptr) {
    return;
  }
#endif

  /* ---- 步骤 2：初始化 ---- */
  REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling);

  /* ---- 步骤 3：设置左矩阵 A 与右矩阵 B ---- */
  mm.SetTensorA(aGlobal[offsetA]);
  mm.SetTensorB(bGlobal[offsetB]);
  mm.SetTail(tailM, tailN);

  /* ---- 步骤 4：完成本核负责的全部矩阵乘 ---- */
  mm.IterateAll(cGlobal[offsetC]);

  /* ---- 步骤 5：结束 ---- */
  mm.End();
}


### 12.6 Host 侧：生成 Tiling、申请 workspace

主机侧比前面多了三件事：

1. **生成 Tiling**：`MultiCoreMatmulTiling` 依据平台信息与矩阵形状算出 `TCubeTiling`，序列化后拷到设备内存；
2. **申请 workspace**：大小由 `GetLibApiWorkSpaceSize()` 给出；
3. **把 float32 的 $A$、$B$ 量化成 half**：用 ACL 提供的 `aclFloatToFloat16`。真值随后按量化后的数据重新计算，理由见 12.5 节。

核数用 `GetCoreNumAic()` 获取——纯 Cube 模式下参与计算的是 AI Core 中的 Cube 核，与前面几节用矢量核数的口径不同。


In [ ]:
%%writefile -a src_matmul/ascendc_matmul_cube.asc
/* ============================================================
 *                       Host 侧
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u;
  return static_cast<float>(state >> 8) * (2.0f / 16777216.0f) - 1.0f;
}

static void GenerateInput(std::vector<float> &a, uint32_t seed) {
  uint32_t s = seed;
  for (size_t i = 0; i < a.size(); ++i) a[i] = LcgNextFloat(s);
}

/* 生成 Tiling：主机侧调用 Tiling 库，序列化到 buf。见 12.4 节 */
static bool GenerateTiling(uint32_t usedCoreNum, std::vector<uint8_t> &buf) {
  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();
  matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);
  cubeTiling.SetDim(static_cast<int32_t>(usedCoreNum));
  cubeTiling.SetAType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::ND,
                      matmul_tiling::DataType::DT_FLOAT16);
  cubeTiling.SetBType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::ND,
                      matmul_tiling::DataType::DT_FLOAT16);
  cubeTiling.SetCType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::ND,
                      matmul_tiling::DataType::DT_FLOAT);
  cubeTiling.SetBiasType(matmul_tiling::TPosition::GM,
                         matmul_tiling::CubeFormat::ND,
                         matmul_tiling::DataType::DT_FLOAT);
  cubeTiling.SetOrgShape(M_SIZE, N_SIZE, K_SIZE);
  cubeTiling.SetShape(M_SIZE, N_SIZE, K_SIZE);
  cubeTiling.SetBufferSpace(-1, -1, -1);
  cubeTiling.EnableBias(false);

  optiling::TCubeTiling tilingData;
  if (cubeTiling.GetTiling(tilingData) == -1) {
    std::printf("[FATAL] Matmul Tiling 生成失败\n");
    return false;
  }
  buf.resize(tilingData.GetDataSize());
  tilingData.SaveToBuffer(buf.data(), buf.size());
  return true;
}

int32_t main() {
  const size_t nA = static_cast<size_t>(M_SIZE) * K_SIZE;
  const size_t nB = static_cast<size_t>(K_SIZE) * N_SIZE;
  const size_t nC = static_cast<size_t>(M_SIZE) * N_SIZE;

  /* ---------- 造数，并量化到 half ---------- */
  std::vector<float> hostA(nA), hostB(nB), hostC(nC);
  GenerateInput(hostA, 2026u);
  GenerateInput(hostB, 20260521u);

  std::vector<aclFloat16> halfA(nA), halfB(nB);
  for (size_t i = 0; i < nA; ++i) halfA[i] = aclFloatToFloat16(hostA[i]);
  for (size_t i = 0; i < nB; ++i) halfB[i] = aclFloatToFloat16(hostB[i]);

  /* 真值按量化之后的数据以 double 计算：本节要检验的是 Cube 把它拿到的
   * 那份数据算得准不准，而不是 half 相对 float32 损失了多少。见 12.5 节 */
  std::vector<float> qa(nA), qb(nB);
  for (size_t i = 0; i < nA; ++i) qa[i] = aclFloat16ToFloat(halfA[i]);
  for (size_t i = 0; i < nB; ++i) qb[i] = aclFloat16ToFloat(halfB[i]);

  std::vector<double> golden(nC, 0.0);
  for (uint32_t i = 0; i < M_SIZE; ++i) {
    for (uint32_t k = 0; k < K_SIZE; ++k) {
      const double av = qa[static_cast<size_t>(i) * K_SIZE + k];
      for (uint32_t j = 0; j < N_SIZE; ++j) {
        golden[static_cast<size_t>(i) * N_SIZE + j] +=
            av * qb[static_cast<size_t>(k) * N_SIZE + j];
      }
    }
  }
  double maxAbsC = 0.0;
  for (size_t i = 0; i < nC; ++i) {
    const double v = std::fabs(golden[i]);
    if (v > maxAbsC) maxAbsC = v;
  }
  const double atol =
      ATOL_COEF * F32_EPS * std::sqrt(static_cast<double>(K_SIZE)) * maxAbsC;

  /* ---------- 初始化设备 ---------- */
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));

  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();
  const uint32_t aicNum = ascendcPlatform->GetCoreNumAic();

  std::vector<uint8_t> tilingBuf;
  if (!GenerateTiling(aicNum, tilingBuf)) return 1;
  const size_t wsSize =
      static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());

  /* ---------- 申请显存并搬入 ---------- */
  const size_t bytesA = nA * sizeof(aclFloat16);
  const size_t bytesB = nB * sizeof(aclFloat16);
  const size_t bytesC = nC * sizeof(float);
  uint8_t *aDev = nullptr, *bDev = nullptr, *cDev = nullptr;
  uint8_t *wsDev = nullptr, *tilingDev = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&aDev, bytesA, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&bDev, bytesB, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&cDev, bytesC, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&wsDev, wsSize, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&tilingDev, tilingBuf.size(),
                        ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(aDev, bytesA, halfA.data(), bytesA,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(aclrtMemcpy(bDev, bytesB, halfB.data(), bytesB,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  ACL_CHECK(aclrtMemcpy(tilingDev, tilingBuf.size(), tilingBuf.data(),
                        tilingBuf.size(), ACL_MEMCPY_HOST_TO_DEVICE));

  /* ---------- 运行一次并校验 ---------- */
  matmul_cube<<<aicNum, nullptr, g_stream>>>(aDev, bDev, cDev, wsDev,
                                             tilingDev);
  ACL_CHECK(aclrtSynchronizeStream(g_stream));
  ACL_CHECK(
      aclrtMemcpy(hostC.data(), bytesC, cDev, bytesC, ACL_MEMCPY_DEVICE_TO_HOST));

  double maxAbs = 0.0;
  for (size_t i = 0; i < nC; ++i) {
    const double e = std::fabs(static_cast<double>(hostC[i]) - golden[i]);
    if (e > maxAbs) maxAbs = e;
  }
  const bool ok = (maxAbs <= atol);
  std::printf(
      "[VERIFY] ver=v4 atol=%.3e max_abs_err=%.3e margin=%.2f result=%s\n",
      atol, maxAbs, atol / (maxAbs > 0.0 ? maxAbs : atol), ok ? "PASS" : "FAIL");

  /* ---------- 计时 ---------- */
  for (int w = 0; w < WARMUP; ++w) {
    matmul_cube<<<aicNum, nullptr, g_stream>>>(aDev, bDev, cDev, wsDev,
                                               tilingDev);
    ACL_CHECK(aclrtSynchronizeStream(g_stream));
  }
  const double t0 = GetTimeMs();
  for (int r = 0; r < REPEAT; ++r) {
    matmul_cube<<<aicNum, nullptr, g_stream>>>(aDev, bDev, cDev, wsDev,
                                               tilingDev);
    ACL_CHECK(aclrtSynchronizeStream(g_stream));
  }
  const double kernelMs = (GetTimeMs() - t0) / REPEAT;
  const double gflop = 2.0 * M_SIZE * N_SIZE * K_SIZE / 1e9;

  std::printf(
      "[PERF]   ver=v4 m=%u n=%u k=%u aic=%u kernel_ms=%.4f gflops=%.2f\n",
      M_SIZE, N_SIZE, K_SIZE, aicNum, kernelMs, gflop / (kernelMs / 1000.0));

  ACL_CHECK(aclrtFree(tilingDev));
  ACL_CHECK(aclrtFree(wsDev));
  ACL_CHECK(aclrtFree(cDev));
  ACL_CHECK(aclrtFree(bDev));
  ACL_CHECK(aclrtFree(aDev));
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());

  std::printf(ok ? "[SUCCESS] Cube 版本通过校验。\n"
                 : "[FAILED] Cube 版本未通过校验！\n");
  return ok ? 0 : 1;
}


### 12.7 编译

Cube 版本由单独一条 `bisheng` 命令生成。与第 9 节的编译命令相比多了三样东西：

- **四个链接库**。官方规定：使用高阶 API 时必须链接 Tiling 相关的库。
- **两个系统库 `-lm` 与 `-ldl`**。它们是上面几个库间接需要的：`-lm` 供主机侧的 `std::sqrt`（推导容差时用到），`-ldl` 供 Tiling 库内部的 `dlopen`。不同机器上链接器的行为略有差别，两个都带上即可。
- **两个编译宏 `HAVE_WORKSPACE` 与 `HAVE_TILING`**。它们让框架自动设置系统 workspace，理由见 12.4 节。


In [ ]:
# Cube 版本单独编译。与 §9 相比多了三样东西：
#   1. Matmul 高阶 API 所需的链接库——官方规定使用高阶 API 时必须链接；
#   2. 两个系统库——它们由上面那几个库间接需要，缺哪个取决于机器上的链接器行为；
#   3. HAVE_WORKSPACE / HAVE_TILING 两个编译宏——官方对 Kernel 直调工程的建议做法，
#      开启后框架自动设置系统 workspace，核函数里不必再调用已废弃的 SetSysWorkspace。
SRC_CUBE = "src_matmul/ascendc_matmul_cube.asc"
EXE_CUBE = "src_matmul/ascendc_matmul_cube"

# 官方要求：使用高阶 API 时必须链接以下库
CUBE_LIBS = ["-ltiling_api", "-lregister", "-lgraph_base", "-lplatform"]

# 系统库。两者都是标准库，在两类机器上都可以带着，不会有副作用：
#   -lm   主机侧的 std::sqrt（推导容差时用到）在部分环境下需要显式链接
#   -ldl  Tiling 库内部通过 dlopen 加载平台信息，部分环境下需要显式链接
CUBE_SYS_LIBS = ["-lm", "-ldl"]

# 编译宏：让框架自动设置系统 workspace，见 12.4 节
CUBE_DEFS = ["-DHAVE_WORKSPACE", "-DHAVE_TILING"]

cmd = ["bisheng", SRC_CUBE] + FLAGS + CUBE_DEFS + CUBE_LIBS + CUBE_SYS_LIBS + ["-o", EXE_CUBE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg[-3000:])

cube_ok = proc.returncode == 0
print("✅ 编译成功" if cube_ok else "❌ 编译失败（返回码 %d）" % proc.returncode)


### 12.8 对照：两条路径的差距

把 Cube 版本的吞吐与前面矢量版本的三档并排放在一起。**这不是一次受控对比**，三处差别要一并说清楚：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 差别 | 说明 |
| --- | --- |
| 数据类型不同 | v1–v3 为 float32，v4 为 half 输入、float32 累加。官方指出 Cube 计算场景 float 算力为 half 算力的 1/4，因此换成 float32 差距会缩小，但不会改变量级 |
| 核数不同 | v3 用的是矢量核，v4 用的是 Cube 核，两者数量由硬件规格决定，不是可以随意对齐的参数。下面的输出会同时给出折合到单核的数值 |
| 误差口径不同 | v4 的真值按量化之后的 $A$、$B$ 重算，两者的误差不可直接比较，理由见 12.5 节 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">差别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据类型不同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1–v3 为 float32，v4 为 half 输入、float32 累加。官方指出 Cube 计算场景 float 算力为 half 算力的 1/4，因此换成 float32 差距会缩小，但不会改变量级</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核数不同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 用的是矢量核，v4 用的是 Cube 核，两者数量由硬件规格决定，不是可以随意对齐的参数。下面的输出会同时给出折合到单核的数值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">误差口径不同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v4 的真值按量化之后的 $A$、$B$ 重算，两者的误差不可直接比较，理由见 12.5 节</td>
</tr>
</tbody>
</table>

因此这张图回答的不是「同样条件下 Cube 快多少倍」，而是一个更实际的问题：**在昇腾上做矩阵乘，按矢量单元的自然写法与按 CANN 主推的写法，最终能拿到的吞吐差多少。** 这也正是本实验要回答的问题。


In [ ]:
if not cube_ok:
    print("Cube 版本未编译成功，跳过本节。")
else:
    out_cube = run_demo(exe=EXE_CUBE)
    print(out_cube)

    rows_cube = parse_rows(out_cube, "PERF")
    if rows_cube:
        r4 = rows_cube[0]
        bd3 = int(rows_main[2]["blockDim"])
        aic = int(r4["aic"])
        gf_vec = [r["gflops"] for r in rows_main]
        gf_best = max(gf_vec)

        labels_all = ["v1\nAIV x1", "v2\nAIV x1", "v3\nAIV x%d" % bd3, "v4\nAIC x%d" % aic]
        vals_all = gf_vec + [r4["gflops"]]

        fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=120)
        ax.bar(range(4), vals_all, 0.5, color=[C_BASE, C_KERNEL, C_OPT, "#7A5AA8"])
        for i, v in enumerate(vals_all):
            ax.text(i, v, "%.0f" % v, ha="center", va="bottom", fontsize=9.5)
        ax.axhline(base_main["cpu_blocked_gflops"], color=C_E2E, lw=1.2, ls="--")
        ax.text(3.4, base_main["cpu_blocked_gflops"] * 1.15,
                "CPU blocked", fontsize=8.5, color=C_E2E, ha="right")
        ax.set_xticks(range(4))
        ax.set_xticklabels(labels_all, fontsize=9)
        ax.set_yscale("log")
        ax.set_ylabel("Achieved GFLOPS")
        ax.set_title("Lab 6: vector unit (fp32) vs Cube unit (half in, fp32 accumulate)")
        ax.grid(axis="y", alpha=0.3, which="both")
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        plt.tight_layout()
        plt.show()

        print("整机口径：矢量单元最高 %.2f GFLOPS（v3，%d 个矢量核，float32），"
              "Cube 单元 %.2f GFLOPS（v4，%d 个 Cube 核，half 输入），相差 %.0f 倍。"
              % (gf_best, bd3, r4["gflops"], aic, r4["gflops"] / gf_best))
        print("折合单核：矢量 %.2f GFLOPS/核，Cube %.2f GFLOPS/核，相差 %.0f 倍。"
              % (gf_best / bd3, r4["gflops"] / aic, (r4["gflops"] / aic) / (gf_best / bd3)))

        if r4["kernel_ms"] < 0.1:
            print()
            print("⚠️  Cube 版本的核函数耗时已不足 0.1 ms，与核函数启动的固定开销"
                  "（官方给出的量级为微秒）处于同一量级。")
            print("    这说明在这个规模上，问题对 Cube 来说已经太小：测到的吞吐是被固定开销"
                  "压低之后的值，Cube 的实际能力还要更高。")
            print("    动手练习第 8 题把规模放大，可以看到这个数字继续上升。")


## 13. 结果分析

> 本节给出的是判读方法与定性规律。具体数值与设备型号、CANN 版本、编译选项以及运行时的负载都有关，不同机器上测得的数并不相同；如果这台服务器同时还有其他用户在使用，个别数值还会出现波动。下面每一条都先说明看哪一个量，再说明它应当呈现什么形状，请以自己运行得到的输出为准。

**① 复用有收益，但很快就不再有收益**

第 11 节的扫描应当呈现这样的形状：$B$ 的读取总量严格按 $1/T$ 下降，而耗时先跟着下降，到某一档之后就不再变化。

在收益尚未消失的区间内，搬运是瓶颈，减少搬运就减少耗时；越过某一档之后，搬运量仍在按 $1/T$ 下降，耗时却不再改变——**这说明瓶颈已经不在搬运上，而转移到了计算上**。

把收益消失的那一档与第 3 节按 UB 容量算出的上界 22 相比，是本实验值得做的一次对照：**在矢量单元上，前者通常远小于后者。** 这说明限制本实现的不是片上容量，而是矢量单元自身的算力。**上界不等于瓶颈**——容量核算告诉你能开到多大，实测才告诉你开到多大有用。

**② 矢量单元的三项限制，Cube 消除了其中两项**

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 矢量单元上的限制 | 性质 | Cube 单元有没有消除它 |
| --- | --- | --- |
| 片上容量给出复用系数的上界 | 由第 3 节核算得到 | **不适用**：Cube 的数据不经过 UB，走的是 L1 / L0 这条通路 |
| 一次乘加需要两条指令（<code>Muls</code> 与 <code>Add</code>） | 指令数是节拍的直接因素 | **消除**：一条 Cube 指令完成一个 $16\times16\times16$ 的块 |
| 每次乘加前需要一次标量取值 | $M \times K$ 次 <code>GetValue</code>，矢量与标量流水之间的同步 | **消除**：Cube 的通路上没有标量参与 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">矢量单元上的限制</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">性质</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Cube 单元有没有消除它</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上容量给出复用系数的上界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由第 3 节核算得到</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**不适用**：Cube 的数据不经过 UB，走的是 L1 / L0 这条通路</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一次乘加需要两条指令（<code>Muls</code> 与 <code>Add</code>）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">指令数是节拍的直接因素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**消除**：一条 Cube 指令完成一个 $16\times16\times16$ 的块</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每次乘加前需要一次标量取值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$M \times K$ 次 <code>GetValue</code>，矢量与标量流水之间的同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**消除**：Cube 的通路上没有标量参与</td>
</tr>
</tbody>
</table>

第 12 节的对照把这件事变成了可以看到的数字：**同一个矩阵乘，换一个计算单元，吞吐是量级上的改变，而不是百分比上的改变。**

这也回答了要不要在矢量单元上继续打磨这个问题。矢量版还有一些余量——例如用 `Axpy` 把两条指令合并成一条（动手练习第 2 题），或者把数据类型换成 half——但这些都是倍数级的改进，**不会改变上面任何一条结论的形状**。量级上的差距在计算单元的选择上。

**③ 换了计算单元之后，规模成了新的问题**

第 12 节的输出里有一条容易被略过的信息：**Cube 版本的核函数耗时可能短到与核函数启动的固定开销处于同一量级**（实验五第 2.2 节引用的官方图给出了这个量级）。一旦如此，测到的吞吐就已经被固定开销压低过，不再反映 Cube 本身的能力。

判断方法很直接：把核函数耗时与那个微秒量级的固定开销相比。若两者同量级，说明**问题对这块硬件而言已经太小**。

这也解释了一件事：真实场景中矩阵乘的规模远大于本实验的 $1024^3$，除了模型本身的需要，也是为了**让专用计算单元的算力得到充分利用**。动手练习第 8 题把规模放大，可以看到这个比值继续上升。

> **一条可以推广的判据**：换用更强的计算单元之后，原来合适的问题规模可能变得太小。优化走到这一步，该调整的不再是实现，而是问题的规模或批量。

**④ 精度：三项应当核对**

- **矢量版三个版本的最大绝对误差应当完全相同。** 复用系数与核数都不改变沿 $k$ 的累加顺序，三者在数值上严格等价。若出现差异，说明某个版本实际上改变了累加次序。
- **容差余量**，即第 4.3 节推出的 `atol` 与实测最大误差之比，应当有数倍的余量。余量小于 1 说明公式低估了误差，大于两个数量级则说明安全系数取得过于保守。
- **Cube 版的误差不与上面两项比较。** 它的输入被量化到 half，误差的来源不同，第 12.5 节已经说明。实测中 Cube 版对量化后真值的误差往往小于矢量版对 float32 真值的误差——这不说明 Cube 算得更准，只说明两者衡量的本来就是不同的东西。

---

### 其余几项的判读提示

以下几项不展开，只给出应当看哪一个量：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 还应留意 | 判读方法 |
| --- | --- |
| 多核扩展性 | v2 与 v3 只差核数。矩阵乘的核函数耗时以毫秒计，而启动的固定开销是微秒量级，占比极小，因此扩展性应当接近线性 |
| 加速比的分母 | 分母是这台机器的 CPU。换一台主机，同样的核函数耗时会算出不同的加速比。**跨机器比较请用 GFLOPS** |
| 两份 CPU 基准之差 | 算法与浮点运算条数完全相同，差别只在循环次序与分块。二者的耗时通常相差一到两个数量级，这个倍数就是局部性在 CPU 上的价值 |
| 单核的等效带宽 | 口径是程序请求了多少字节除以时间，不等于有多少字节真的经过了 HBM——$B$ 的一部分重复读取可能命中 L2。动手练习第 5 题给出分辨的办法 |
| 矩阵乘与卸载 | 运算量随规模三次方增长，输入输出只有二次方，因此规模越大，主机与设备之间的传输占比越低。逐元素算子没有这个性质 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">还应留意</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">判读方法</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多核扩展性</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 与 v3 只差核数。矩阵乘的核函数耗时以毫秒计，而启动的固定开销是微秒量级，占比极小，因此扩展性应当接近线性</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">加速比的分母</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分母是这台机器的 CPU。换一台主机，同样的核函数耗时会算出不同的加速比。**跨机器比较请用 GFLOPS**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两份 CPU 基准之差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算法与浮点运算条数完全相同，差别只在循环次序与分块。二者的耗时通常相差一到两个数量级，这个倍数就是局部性在 CPU 上的价值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单核的等效带宽</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">口径是程序请求了多少字节除以时间，不等于有多少字节真的经过了 HBM——$B$ 的一部分重复读取可能命中 L2。动手练习第 5 题给出分辨的办法</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矩阵乘与卸载</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运算量随规模三次方增长，输入输出只有二次方，因此规模越大，主机与设备之间的传输占比越低。逐元素算子没有这个性质</td>
</tr>
</tbody>
</table>

---

### 🎓 结论

本实验的主线是一个选择：**同一个矩阵乘，放在哪个计算单元上算。**

前半程在矢量单元上把能做的都做了——提高复用、用满核数——并测出了这条路径的上限；后半程把同一个算子交给 Cube 单元，用 `Matmul` 高阶 API 实现。两者的差距不是百分之几，而是量级。

由此得到一条可以推广的工作方法：**在动手优化一个算子之前，先确认它是不是跑在合适的计算单元上。** 计算单元选错时，再精细的实现也追不回这个量级；选对之后，第 3 节那套片上核算、第 11 节那套复用分析才重新变得有意义——只不过对象换成了 L1 与 L0，由 `Matmul` 高阶 API 内部去做。

换用计算单元之后还留下一个新问题：本实验的规模已经不足以让 Cube 的算力得到充分利用（见 ③）。**优化不是一条直线，每推进一步，瓶颈都可能转移到别处。**

矢量单元这一程同样有其价值：**它把「为什么需要 Cube」从一条需要记忆的结论，变成了一次可以自己测出来的结果。**


## 14. 🔧 动手练习

1. **确认片上占用检查生效，并判断这项核算的意义。** 用 `-DLAB_TILE_M=24` 编译运行，程序会报 `[FATAL] 片上占用 204 KB，超出 UB 的 192 KB`。结合第 11 节测出的收益消失的那一档回答：第 3 节那套片上占用核算，在本实验的规模下是否影响了最终性能？在什么条件下它会变得关键？
2. **把两条指令合并为一条。** 把内层换成 `AscendC::Axpy`（`tmpBuf` 随之可以取消）。先预测单核 GFLOPS 会提高多少倍，**再测量核对**。若实测明显达不到预测的倍数，请判断多出来的开销在哪里（提示：`GetValue` 这一环并未被消除）。
3. **改变 $K$，检验容差公式。** 令 $K$ 取 256、512、1024、2048，记录 `atol` 与 `max_abs_err`。实测误差是否按 $\sqrt{K}\max|C|$ 增长？余量是否稳定在同一量级？
4. **换一个切分方向。** 现在沿 $M$ 切分，每个核读完整的 $B$。改成沿 $N$ 切分（每核负责 $C$ 的若干列、读完整的 $A$），重新核算搬运量并测量。**先根据第 13 节末尾关于多核扩展性的那条提示预测结果会不会变，再动手。**
5. **让 $B$ 装不进 L2。** 把 $N$、$K$ 加大到 $B$ 明显超过设备的 L2 容量，重测 `TILE_M = 1` 那一档的等效带宽，与原先的值相比。若明显下降，说明原先确实有相当一部分重复读取命中了 L2——这是分辨第 13 节末尾那条等效带宽提示中两种情形的办法。
6. **核数扫描。** 用 `-DLAB_BLOCK_DIM` 依次取 1、2、4、8、16、32 重新编译运行 v3，记录核函数耗时。加速比是否接近线性？在核数很多、单核计算量很小时，为什么增长会变慢？请结合第 13 节末尾关于多核扩展性的那条提示解释。
7. **【进阶】把 $C$ 也分块。** 现在 $C$ 的一块占 `TILE_M × N`，$N$ 一大就放不下。改成同时沿 $N$ 分块（块大小 `TILE_N`），使片上占用与 $N$ 无关。此时计算访存比的表达式变成什么？（提示：不再是 $T/2$。）
8. **把 Cube 版本的规模放大。** 第 13 节 ③ 指出，本实验的规模对 Cube 来说可能已经太小。把第 12 节的 $M$、$N$、$K$ 依次改成 2048、4096，记录核函数耗时与达到的 GFLOPS。**两件事要一起观察**：一是达到的吞吐是否随规模继续上升，直到固定开销不再占主导；二是矢量版换一个形状要重新核算片上占用并重编译，而 Cube 版只要把新的形状交给 Tiling 库。


## 15. 🤔 思考题

- 本实验的运算量是 $2MNK$，输入输出只有 $MK + KN + MN$。**请据此说明：为什么矩阵乘的计算访存比可以随规模增长，而逐元素算子不能？**
- 第 11 节的曲线在某一档之后转平。**转平这件事本身说明了什么？** 它与实验五中的融合收益边界是同一类现象吗？
- 复用系数的容量上界由 UB 大小决定，而收益消失的那一档由搬运与计算的相对速度决定，二者互不相干。**如果换一台带宽只有现在四分之一、算力不变的机器，这两个量各会怎样变化？** 此时第 3 节的核算会不会变得关键？
- 第 13 节末尾的提示指出，单核的等效带宽无法区分 HBM 与 L2。**除练习 5 那种把数据规模撑大的办法外，还有别的实验设计可以把两者分开吗？**
- 矢量版三个版本的最大绝对误差完全相同，因为累加顺序未变。**如果沿 $K$ 方向也做多核切分（每核负责一段 $k$，最后把部分和相加），误差会怎样变化？** 变大还是变小？（提示：回顾实验三关于累加链长度的结论。）
- 本实验用绝对误差判定，而误差随输出量级缩放的算子适合用相对误差判定。**请归纳：选择判定口径的依据是什么？**
- 第 12.5 节指出，Cube 版用 half 输入，误差来源与 float32 版不同，两者的容差不能直接比较。那么在什么条件下，两个不同实现的容差**是**可以比较的？
- Cube 单元把「标量乘向量再累加」这一固定模式硬件化。**哪些别的算子也能受益？哪些不能？** 请各举一例并说明判断依据。
- 第 12.4 节指出，矢量版的复用系数是写死的模板实参，Cube 版的切分参数由 Tiling 库在运行期算出。**这两种做法各有什么代价？** 在什么场景下前者反而更合适？
- Cube 把算力整体抬高之后，搬运重新成为瓶颈的位置右移，提高复用重新变得重要。**请据此推断：`Matmul` 高阶 API 内部为什么需要 L1 与 L0 两级分块，而不是像本实验的矢量版这样只用一级？**


## 16. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 矩阵乘与前五个实验的差别 | 每个输入元素可以被反复使用；优化的方向不是减少必须搬运的数据，而是提高已搬入数据的复用次数 |
| 计算访存比 | 官方用语，即单位搬运量对应的计算量；本实验取 FLOP/Byte 为单位。逐元素算子固定在 0.5 附近，矩阵乘由实现决定 |
| 秩一更新 | 取 $A$ 的第 $k$ 列与 $B$ 的第 $k$ 行做外积、累加到 $C$；一次在片上累加的行数即复用系数 |
| 复用系数的容量上界 | 由 UB 容量核算得到；**上界不等于瓶颈**，实际有收益的范围通常远小于它 |
| 收益的消失 | 搬运量仍在下降而耗时不再下降，说明瓶颈已由搬运转到计算；此后提高复用不再有效 |
| 队列深度按搬运频率选 | 只有被流水起来的那一段搬运才需要双缓冲；一次性搬入的数据用深度 1 |
| 队列也负责同步 | <code>EnQue</code> 紧跟 <code>DeQue</code> 不是排队，而是取得 MTE2 与矢量流水之间的一次同步 |
| **矢量单元的三项限制** | 片上容量给出复用系数上界；一次乘加要两条指令；每次乘加前要一次标量取值 |
| **Cube 单元的片上通路** | GM → L1（转分形格式）→ L0A / L0B → Cube → L0C（float32 累加）→ FixPipe → GM，**不经过 UB，也没有标量参与** |
| **一条 Cube 指令的规模** | fp16 输入时 1 cycle 可算 $16\times16\times16$ 个数，即一个 4096 次乘加的块 |
| **<code>Matmul</code> 高阶 API 的五步** | 创建对象 → 初始化 → 设置 A/B → <code>IterateAll</code> → <code>End</code>；分块与格式转换都在 API 内部 |
| **切分参数的来源** | 矢量版写死在源码里，换形状要重编译；Cube 版由 Tiling 库在运行期依据平台与形状算出 |
| 容差随规模缩放 | 矩阵乘的每个输出是 $K$ 项累加，容差按 $c\,\varepsilon\sqrt{K}\,\lVert C\rVert_\infty$ 推导，不用固定常数 |
| 判定口径 | 误差不随输出量级缩小时用绝对误差；随输出量级缩放时用相对误差。**口径跟随误差的来源，不跟随输出的量级** |
| 跨机器比较用 GFLOPS | 加速比的分母是这台机器的 CPU，换机器就会变；GFLOPS 是绝对吞吐 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矩阵乘与前五个实验的差别</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个输入元素可以被反复使用；优化的方向不是减少必须搬运的数据，而是提高已搬入数据的复用次数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计算访存比</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方用语，即单位搬运量对应的计算量；本实验取 FLOP/Byte 为单位。逐元素算子固定在 0.5 附近，矩阵乘由实现决定</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">秩一更新</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">取 $A$ 的第 $k$ 列与 $B$ 的第 $k$ 行做外积、累加到 $C$；一次在片上累加的行数即复用系数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">复用系数的容量上界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 UB 容量核算得到；**上界不等于瓶颈**，实际有收益的范围通常远小于它</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">收益的消失</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">搬运量仍在下降而耗时不再下降，说明瓶颈已由搬运转到计算；此后提高复用不再有效</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">队列深度按搬运频率选</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只有被流水起来的那一段搬运才需要双缓冲；一次性搬入的数据用深度 1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">队列也负责同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>EnQue</code> 紧跟 <code>DeQue</code> 不是排队，而是取得 MTE2 与矢量流水之间的一次同步</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**矢量单元的三项限制**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上容量给出复用系数上界；一次乘加要两条指令；每次乘加前要一次标量取值</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**Cube 单元的片上通路**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">GM → L1（转分形格式）→ L0A / L0B → Cube → L0C（float32 累加）→ FixPipe → GM，**不经过 UB，也没有标量参与**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**一条 Cube 指令的规模**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">fp16 输入时 1 cycle 可算 $16\times16\times16$ 个数，即一个 4096 次乘加的块</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**<code>Matmul</code> 高阶 API 的五步**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">创建对象 → 初始化 → 设置 A/B → <code>IterateAll</code> → <code>End</code>；分块与格式转换都在 API 内部</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**切分参数的来源**</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矢量版写死在源码里，换形状要重编译；Cube 版由 Tiling 库在运行期依据平台与形状算出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">容差随规模缩放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">矩阵乘的每个输出是 $K$ 项累加，容差按 $c\,\varepsilon\sqrt{K}\,\lVert C\rVert_\infty$ 推导，不用固定常数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">判定口径</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">误差不随输出量级缩小时用绝对误差；随输出量级缩放时用相对误差。**口径跟随误差的来源，不跟随输出的量级**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">跨机器比较用 GFLOPS</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">加速比的分母是这台机器的 CPU，换机器就会变；GFLOPS 是绝对吞吐</td>
</tr>
</tbody>
</table>

### 两条贯穿本章的原则

> **在动手优化一个算子之前，先确认它是不是跑在合适的计算单元上。** 计算单元选错时，再精细的实现也追不回这个量级。

> **上界不等于瓶颈。** 容量核算告诉你能开到多大，实测才告诉你开到多大有用。两者都要做，但不能相互替代。

### 与后续实验的衔接

➡️ **后续内容：Softmax**。本实验的两程都在回答同一个问题——**怎样算得更快**：先把矢量单元的实现优化到上限，再换用为矩阵乘设计的计算单元。Softmax 要回答的是另一个问题：**怎样保证算得对**。

Softmax 的算式中含有指数函数，输入的幅度稍大，float32 就会溢出，因此必须先把算式改写成数值稳定的形式，才谈得上性能。判定的口径也要随之重新设计：本实验用一条绝对误差就够了，而 Softmax 会看到，只用一条判据会把一个完全错误的结果判为通过。

**两个实验合起来给出算子开发的两条主线：性能取决于把计算放在哪个单元上，正确性取决于把算式写成什么形式。**
